<span style='font-size:x-large'>**Text Data Cleani**</span><span style='font-size:small'><span style='font-size:x-large'>**ng**</span></span>

<span style='font-size:small'>This notebook covers the first stage of our capstone project: loading and cleaning the dataset. Since the dataset is already complete, we will focus on standardising and formatting the text, removing unnecessary characters, and preparing it for preprocessing and topic modelling.</span>

<span style='font-size:large'>**Relevant Lectures:**</span>

<span style='font-size:small'>Course 2, Module 13: "Text Analysis for Environmental Data " by Luke Sanford — Covers core techniques for analysing environmental text—including cleaning, tokenisation, stop\-word removal, word\-frequency analysis, sentiment analysis, and building custom dictionaries.</span>

<span style='font-size:small'>Course 3, Module 8: "Using EDA to Clean Data" by Angel — Discusses identifying common data issues and addressing them.</span>


In [3]:
# Load the required packages
library(tidyverse)     # Data manipulation and visualization
library(tidytext)      # Text mining with tidy data principles
library(SnowballC)     # Stemming
library(tm)            # Text mining utilities
library(dplyr)         # Data manipulation
library(stringr)       # String manipulation
library(janitor)       # data cleaning and standardizing

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: NLP




Attaching package: ‘NLP’




The following object is masked from ‘package:ggplot2’:

    annotate





Attaching package: ‘janitor’




The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test




In [4]:
#lets look into our data set
docs <- read_csv ("data/raw/raw_data.csv", locale = locale(encoding = "latin1"))
#check the structure
str(docs)
#check rows
head(docs,5)

Rows: 5 Columns: 5


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): doc.id, doc_name, source, text
dbl (1): Year



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


spc_tbl_ [5 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ doc.id  : chr [1:5] "doc_01" "doc_02" "doc_03" "doc_04" ...
 $ doc_name: chr [1:5] "Carbon Profits or Pastoralist Precarity" "Blood_Carbon_Report" "Cows, Carbon and Conflict" "01 January 2021 Ð 31 December 2021 Monitoring report" ...
 $ source  : chr [1:5] "Independent" "Independent" "Independent" "official" ...
 $ Year    : num [1:5] 2025 2023 2025 2024 2023
 $ text    : chr [1:5] "The NKRCP encompasses 1,993,075 hectares (4.7 million acres) of grassland in northern Kenya (Pinette, 2024). It"| __truncated__ "7. Rights to carbon, and benefits from the project7.1 Consent, consultation and information provision The evide"| __truncated__ "In KenyaÕs rugged north, where the effects of climate change are vivid and dire, Northern Rangelands Trust (NRT"| __truncated__ "4.1 Net Positive Community Impacts 4.1.1 Community Impacts (CCB, CM2.1)\nThe project provided substantial econo"| __truncated__ ...
 - attr(*, "spec")=
  .. cols(
  ..

In [5]:
#get details about the dataset
docs %>%
  mutate(word_count = str_count(text, "\\S+")) %>%
  group_by(source) %>%
  summarise(
    n_docs = n(),
    total_words = sum(word_count),
    avg_words = mean(word_count)
  )

source,n_docs,total_words,avg_words
<chr>,<int>,<int>,<dbl>
Independent,3,9814,3271.333
official,2,8405,4202.500


In [7]:
# data cleaning and standardizing our column names
clean_docs <- docs %>%
  clean_names() %>%
  filter(!is.na(text))%>%
  mutate(text =text %>%
           str_remove_all("\\d+")%>%
           str_replace_all("[[:punct:]]","")%>%
           str_remove_all("\uFFFD")%>%
           tolower()%>%
           str_squish(),
        )
    print(clean_docs)
# Save cleaned data
write_csv(clean_docs, "data/cleaned/clean_docs.csv")

# A tibble: 5 × 5
  doc_id doc_name                                             source  year text 
  <chr>  <chr>                                                <chr>  <dbl> <chr>
1 doc_01 Carbon Profits or Pastoralist Precarity              Indep…  2025 the …
2 doc_02 Blood_Carbon_Report                                  Indep…  2023 righ…
3 doc_03 Cows, Carbon and Conflict                            Indep…  2025 in k…
4 doc_04 01 January 2021 Ð 31 December 2021 Monitoring report offic…  2024 net …
5 doc_05 2017 Ð 2020 Monitoring report                        offic…  2023 comm…
